In [2]:
!pip install -q xgboost imbalanced-learn

import pandas as pd
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [3]:
uploaded = files.upload()

df = pd.read_csv("creditcard.csv")

print("Dataset shape:", df.shape)
print(df.head())

Saving creditcard.csv to creditcard.csv
Dataset shape: (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.79827

In [4]:
X = df.drop("Class", axis=1)
y = df["Class"]

print("Normal transactions:", sum(y == 0))
print("Fraud transactions:", sum(y == 1))

Normal transactions: 284315
Fraud transactions: 492


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

smote = SMOTE(
    sampling_strategy=0.3,
    random_state=42
)

X_train, y_train = smote.fit_resample(
    X_train, y_train
)

print("After SMOTE:")
print(y_train.value_counts())

After SMOTE:
Class
0    227451
1     68235
Name: count, dtype: int64


In [6]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    eval_metric="auc",
    random_state=42
)

model.fit(X_train, y_train)

probability = model.predict_proba(X_test)[:, 1]

print("XGBoost ROC-AUC:",
      round(roc_auc_score(y_test, probability), 4))

XGBoost ROC-AUC: 0.9775


In [7]:
threshold = 0.30

prediction = (probability >= threshold).astype(int)

print(classification_report(
    y_test,
    prediction
))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.39      0.90      0.54        98

    accuracy                           1.00     56962
   macro avg       0.69      0.95      0.77     56962
weighted avg       1.00      1.00      1.00     56962



In [8]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Top 10 important features:")
print(importance.head(10))

Top 10 important features:
V14    0.590316
V12    0.054538
V10    0.043054
V4     0.038898
V17    0.037714
V3     0.020536
V13    0.015148
V8     0.013658
V9     0.012148
V28    0.011370
dtype: float32
